# Stage 1 — Smoke Test: Persian Watermarking
**Project: Persian watermarking on speech synthesized with MMS-TTS + AudioSeal**

**What is the goal of this stage?**
To prove just one thing: in Colab, can we synthesize Persian speech, put an AudioSeal watermark on it, and find it again?
We do not test attacks and full robustness yet. That is stage 2.

**What do you send back?**
1. The file `summary_step1.csv`
2. The zip file `persian-watermark-step1.zip` (clean + watermarked audio)
3. Answers to the two questions at the end of the notebook: was the Persian speech natural? Did the watermarked audio sound different from the clean one?

**Colab for a first-timer — very short:**
- Each grey block is a **cell**.
- To run a cell, click on it and press the ▶ button on its left, or press `Shift + Enter`.
- Run the cells **in order, from top to bottom**. If one cell has not finished installing, do not run the next one.
- A green tick means success, a red cross means an error — send me the error text.
- If Colab disconnects and you lose the runtime, just run this same notebook again from the top. Each notebook has its own environment.


## Cell 1 — Environment check: do we have a GPU?
First, let's see what Python and GPU we have. If there is no GPU, go to the top menu:
select `Runtime > Change runtime type > T4 GPU`, then run this cell again.


In [ ]:
import sys, torch
print("Python:", sys.version)
print("torch:", torch.__version__)
print("Is CUDA available?", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU")
print("GPU count:", torch.cuda.device_count())


## Cell 2 — Install
This cell installs `transformers` for Persian MMS-TTS, `audioseal` for watermarking, and `uroman` to convert Persian letters.
It takes a few minutes. Wait for the green tick.


In [ ]:
!pip install -q transformers audioseal soundfile uroman
print("Install done")


## Cell 3 — Check the install
If this cell errors, cell 2 has not finished. Run it again first.


In [ ]:
import transformers, audioseal, soundfile
print("transformers:", transformers.__version__)
print("audioseal installed")
print("soundfile installed")
import torch
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


## Cell 4 — The 10 fixed Persian sentences
These same 10 sentences stay fixed in every stage. Do not change a sentence, or the numbers will no longer be comparable.
The sentences are short and long, statements and commands, so the speech is varied.


In [ ]:
# The 10 fixed Persian test sentences — kept in Persian in every stage so the numbers stay comparable
SENTENCES = [
    "سلام، حالت چطور است؟",
    "هوا امروز در تهران آفتابی است.",
    "من هر روز صبح کتاب می‌خوانم.",
    "هوش مصنوعی دارد زندگی ما را تغییر می‌دهد.",
    "این آزمایش برای بررسی واترمارک فارسی انجام می‌شود.",
    "لطفا در را ببند و چراغ را خاموش کن.",
    "دانشجویان در آزمایشگاه روی پروژه گفتار کار می‌کنند.",
    "صدای ساخته‌شده باید طبیعی و قابل فهم باشد.",
    "ما می‌خواهیم بفهمیم واترمارک روی فارسی هم می‌ماند یا نه.",
    "فردا صبح ساعت نه به دانشگاه می‌روم.",
]
for i, s in enumerate(SENTENCES, 1):
    print(f"{i:02d}. {s}  ({len(s.split())} words)")
print("\nSentence count:", len(SENTENCES))
print("Total words:", sum(len(s.split()) for s in SENTENCES))


## Cell 5 — Load Persian MMS-TTS
Model: `facebook/mms-tts-fas`
- `fas` is the ISO 639-3 code for Persian, not `fa`.
- This model is VITS and synthesizes the waveform directly, at 16 kHz.
- If the tokenizer needs uroman, it applies it itself.


In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "facebook/mms-tts-fas"

print("Loading:", MODEL_ID, "on", DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Does the tokenizer need uroman?", getattr(tokenizer, "is_uroman", "unknown"))
tts = VitsModel.from_pretrained(MODEL_ID).to(DEVICE)
tts.eval()
SR = tts.config.sampling_rate
print("Model ready. sampling_rate =", SR, "Hz")


## Cell 6 — Single-sentence test: synthesize only sentence 1
Before building all 10 sentences, build just one and listen to it. If this one is broken, do not build the rest.


In [ ]:
import os, soundfile as sf
from IPython.display import Audio, display

os.makedirs("persian_tts_clean", exist_ok=True)

def synth_fa(text):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        wav = tts(**inputs).waveform
    return wav[0].cpu().numpy(), SR

wav1, sr = synth_fa(SENTENCES[0])
print("Sentence 1:", SENTENCES[0])
print("Audio length:", round(len(wav1)/sr, 2), "seconds | samples:", len(wav1))
sf.write("persian_tts_clean/utt_001_clean.wav", wav1, sr)
display(Audio(wav1, rate=sr))
print("Saved: persian_tts_clean/utt_001_clean.wav")


## Cell 7 — Synthesize all 10 clean sentences
Now build all 10 sentences and save them. Output: `persian_tts_clean/utt_001_clean.wav` to `utt_010_clean.wav`


In [ ]:
import numpy as np, soundfile as sf, time

durations = []
t0 = time.time()
for i, text in enumerate(SENTENCES, 1):
    wav, sr = synth_fa(text)
    path = f"persian_tts_clean/utt_{i:03d}_clean.wav"
    sf.write(path, wav, sr)
    d = len(wav)/sr
    durations.append(d)
    print(f"{i:02d}. {d:.2f}s  {path}")

print(f"\nTotal audio duration: {sum(durations):.2f} seconds")
print(f"Mean per sentence: {np.mean(durations):.2f} seconds")
print(f"Synthesis time (wall-clock): {time.time()-t0:.1f} seconds")


## Cell 8 — Listen: sentences 1, 5 and 10
Listen to these three and note: is the Persian natural? Was any word misread?


In [ ]:
from IPython.display import Audio, display
import soundfile as sf
for i in [1, 5, 10]:
    wav, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    print(f"--- Sentence {i}: {SENTENCES[i-1]}")
    display(Audio(wav, rate=sr))


## Cell 9 — Load AudioSeal
- Generator: `audioseal_wm_16bits` — builds a 16-bit watermark.
- Detector: `audioseal_detector_16bits`
- Both work at 16 kHz, exactly the MMS-TTS rate. So no resampling is needed.


In [ ]:
from audioseal import AudioSeal

print("Loading AudioSeal generator...")
generator = AudioSeal.load_generator("audioseal_wm_16bits").to(DEVICE)
generator.eval()
print("Loading AudioSeal detector...")
detector = AudioSeal.load_detector("audioseal_detector_16bits").to(DEVICE)
detector.eval()
print("AudioSeal ready.")

# One fixed 16-bit message for all sentences — so it stays comparable in stage 2
MESSAGE = torch.tensor([[1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0]], dtype=torch.int32, device=DEVICE)
print("Fixed 16-bit message:", MESSAGE.cpu().numpy().tolist()[0])


## Cell 10 — Single-sentence watermark test: only sentence 1
First run the detector on the clean audio, then on the watermarked audio.
Expected: probability low on clean audio (near 0), high on watermarked audio (near 1).
If it is the other way round, something is wrong — do not continue.


In [ ]:
import torch, soundfile as sf, numpy as np
from IPython.display import Audio, display

def to_tensor(wav_np):
    return torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

def detect_prob(wav_np):
    wav_t = to_tensor(wav_np)
    with torch.no_grad():
        try:
            result, msg = detector.detect_watermark(wav_t, sample_rate=16000)
        except TypeError:
            result, msg = detector.detect_watermark(wav_t)
    # result may be a number, a scalar tensor, or a multi-dim tensor — convert them all to one number
    if isinstance(result, torch.Tensor):
        prob = float(result.detach().float().mean().cpu().item())
    else:
        prob = float(result)
    if isinstance(msg, torch.Tensor):
        bits = msg.detach().cpu().numpy().reshape(-1).astype(int).tolist()
    else:
        bits = list(msg)
    return prob, bits

def bit_accuracy(decoded):
    true = MESSAGE.cpu().numpy().reshape(-1).astype(int).tolist()
    correct = sum(int(a)==int(b) for a,b in zip(decoded, true))
    return correct, correct/len(true)

wav_clean, sr = sf.read("persian_tts_clean/utt_001_clean.wav")
p_clean, _ = detect_prob(wav_clean)
print(f"Sentence 1 clean: watermark probability = {p_clean:.3f}  (expected: low)")

with torch.no_grad():
    try:
        wm = generator.get_watermark(to_tensor(wav_clean), sample_rate=16000, message=MESSAGE)
    except TypeError:
        wm = generator.get_watermark(to_tensor(wav_clean), message=MESSAGE)
    wav_wm = (to_tensor(wav_clean) + wm).squeeze().cpu().numpy()

os.makedirs("persian_tts_watermarked", exist_ok=True)
sf.write("persian_tts_watermarked/utt_001_wm.wav", wav_wm, sr)
p_wm, bits_wm = detect_prob(wav_wm)
correct, acc = bit_accuracy(bits_wm)
print(f"Sentence 1 watermarked: probability = {p_wm:.3f}  (expected: high)")
print(f"Decoded bits: {bits_wm}")
print(f"Correct bits: {correct}/16  bit accuracy: {acc:.2%}")

print("\n— Listen: clean first, then watermarked —")
display(Audio(wav_clean, rate=sr))
display(Audio(wav_wm, rate=sr))


## Cell 11 — Watermark all 10 sentences + summary table
We watermark all of them with the same 16-bit message and record the clean and watermarked probability for each sentence.
Output: `summary_step1.csv`


In [ ]:
import csv, soundfile as sf

rows = []
for i, text in enumerate(SENTENCES, 1):
    wav_clean, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    p_clean, _ = detect_prob(wav_clean)
    with torch.no_grad():
        try:
            wm = generator.get_watermark(to_tensor(wav_clean), sample_rate=16000, message=MESSAGE)
        except TypeError:
            wm = generator.get_watermark(to_tensor(wav_clean), message=MESSAGE)
        wav_wm = (to_tensor(wav_clean) + wm).squeeze().cpu().numpy()
    sf.write(f"persian_tts_watermarked/utt_{i:03d}_wm.wav", wav_wm, sr)
    p_wm, bits_wm = detect_prob(wav_wm)
    correct, acc = bit_accuracy(bits_wm)
    rows.append({
        "id": i, "text": text,
        "duration_s": round(len(wav_clean)/sr, 2),
        "clean_prob": round(p_clean, 3),
        "wm_prob": round(p_wm, 3),
        "wm_detected_gt_0.5": bool(p_wm > 0.5),
        "bits_correct": correct, "bit_accuracy": round(acc, 3),
    })
    print(f"{i:02d}. clean={p_clean:.3f}  watermarked={p_wm:.3f}  bits={correct}/16")

with open("summary_step1.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print("\nSaved: summary_step1.csv")

import pandas as pd
df = pd.DataFrame(rows)
print(df[["id","duration_s","clean_prob","wm_prob","bits_correct"]].to_string(index=False))
print(f"\nMean clean probability: {df['clean_prob'].mean():.3f}")
print(f"Mean watermarked probability: {df['wm_prob'].mean():.3f}")
print(f"Correctly detected (of 10): {int(df['wm_detected_gt_0.5'].sum())}/10")
print(f"Mean bit accuracy: {df['bit_accuracy'].mean():.2%}")
print(f"Total audio duration: {df['duration_s'].sum():.2f} seconds")


## Cell 11b — Stage-1 chart: clean vs watermarked
This chart is drawn from the same `summary_step1.csv`; no extra computation.
- Left: detection probability per sentence — the clean bar should be low and the watermarked bar high. The dashed 0.5 line is the detection threshold.
- Right: bit accuracy out of 16 bits per sentence.
If one sentence is flipped, it shows right here.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv("summary_step1.csv")
print(df[["id","clean_prob","wm_prob","bits_correct"]].to_string(index=False))

x = np.arange(len(df))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: detection probability, clean vs watermarked
axes[0].bar(x - w/2, df["clean_prob"], w, label="Clean")
axes[0].bar(x + w/2, df["wm_prob"], w, label="Watermarked")
axes[0].axhline(0.5, linestyle="--", label="Threshold 0.5")
axes[0].set_xticks(x); axes[0].set_xticklabels(df["id"])
axes[0].set_xlabel("Sentence id")
axes[0].set_ylabel("Detection probability")
axes[0].set_title("Step 1: Clean vs Watermarked detection")
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Right: bit accuracy
axes[1].bar(df["id"], df["bit_accuracy"])
axes[1].set_xlabel("Sentence id")
axes[1].set_ylabel("Bit accuracy")
axes[1].set_title("Step 1: Bit accuracy (of 16 bits)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("chart_step1.png", dpi=150)
plt.show()
print("\nSaved: chart_step1.png")
print(f"Mean clean probability: {df['clean_prob'].mean():.3f} | Mean watermarked probability: {df['wm_prob'].mean():.3f}")
print(f"Detected: {int(df['wm_detected_gt_0.5'].sum())}/10 | Mean bit accuracy: {df['bit_accuracy'].mean():.2%}")


## Cell 12 — Small MP3 test (sentence 1 only, to preview stage 2)
This is not a final result, just a preview: does the watermark survive MP3 compression at 128 kbps?
Stage 2 repeats this for all 10 sentences and several more attacks.


In [ ]:
import subprocess, soundfile as sf

# Convert watermarked sentence 1 to MP3 and back to WAV
subprocess.run(["ffmpeg","-y","-loglevel","error","-i","persian_tts_watermarked/utt_001_wm.wav","-codec:a","libmp3lame","-b:a","128k","/tmp/utt_001_wm.mp3"], check=True)
subprocess.run(["ffmpeg","-y","-loglevel","error","-i","/tmp/utt_001_wm.mp3","/tmp/utt_001_wm_from_mp3.wav"], check=True)
wav_mp3, sr = sf.read("/tmp/utt_001_wm_from_mp3.wav")
# If the sample rate changed, it does not matter for the detector — detect_prob calls with 16k, so make sure wav is 16k
print("Sample rate after MP3:", sr)
p_mp3, bits_mp3 = detect_prob(wav_mp3)
correct, acc = bit_accuracy(bits_mp3)
print(f"Sentence 1 after MP3 128k: prob = {p_mp3:.3f} | Correct bits: {correct}/16 ({acc:.0%})")
print("If the probability is still above 0.5, the watermark has survived this attack for now.")


## Cell 13 — Zip the outputs for download
Zip the folder, because Colab does not download folders. After running, download the zip from the Files panel on the left.


In [ ]:
import shutil, os
shutil.make_archive("persian-watermark-step1", "zip", root_dir=".", base_dir="persian_tts_clean")
# A separate zip for watermarked files + summary — simpler: build one full zip
import zipfile
with zipfile.ZipFile("persian-watermark-step1.zip", "w") as z:
    for folder in ["persian_tts_clean", "persian_tts_watermarked"]:
        for fn in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, fn))
    z.write("summary_step1.csv")
    import os as _os
    if _os.path.exists("chart_step1.png"):
        z.write("chart_step1.png")
print("Created: persian-watermark-step1.zip")
print("Size:", round(os.path.getsize("persian-watermark-step1.zip")/1024, 1), "KB")
print("Files: 10 clean + 10 watermarked + summary_step1.csv + chart_step1.png")


## Done — what do you send back?
**1. Send me the file `summary_step1.csv`**
**2. Send me the file `persian-watermark-step1.zip` too**

**And answer these two questions:**
- **A) Listening:** was the MMS-TTS Persian speech natural? Did you listen to sentences 1, 5 and 10? Which sentence was read badly?
- **B) Listening comparison:** did watermarked sentence 1 sound different from the clean one? Did you hear anything or not?

**If stage 1 succeeded, that means:**
- Mean clean probability is low (near 0)
- Mean watermarked probability is high (near 1)
- At least 9 of 10 sentences are detected

If not, that is fine too — just send the same numbers; that itself is a result: it means the English-centric AudioSeal is weak on Persian. We will build stage 2 on that basis.
